In [1]:
import pandas as pd

mtcars = pd.read_csv("../data/mtcars.csv")
mtcars.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [2]:
y = mtcars.pop("mpg")
X = mtcars
X.shape

(32, 10)

In [3]:
# Coefficient paths: how Lasso shrinks coefficients to zero as alpha grows
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import lasso_path
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
alphas, coefs, _ = lasso_path(X_scaled, y - y.mean(), alphas=np.logspace(-3, 1, 100))

plt.figure(figsize=(8, 5))
for name, coef in zip(X.columns, coefs):
    plt.plot(alphas, coef, label=name)
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("Coeficiente")
plt.legend(ncol=2, fontsize=8)
plt.grid(color="lightgray", linestyle="--", linewidth=0.5)
plt.show()

<Figure size 800x500 with 1 Axes>

In [4]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("lasso", Lasso(max_iter=10000)),
    ]
)

estimator = GridSearchCV(
    pipeline,
    param_grid={"lasso__alpha": np.logspace(-3, 1, 50)},
    cv=5,
    scoring="neg_mean_squared_error",
)
estimator.fit(X, y)
estimator.best_params_, -estimator.best_score_

({'lasso__alpha': 0.868511373751352}, 10.443711536342837)

In [5]:
# Selected variables (non-zero coefficients)
coefs = pd.Series(
    estimator.best_estimator_.named_steps["lasso"].coef_,
    index=X.columns,
)
coefs.round(3)

cyl    -1.546
disp   -0.000
hp     -0.755
drat    0.000
wt     -2.572
qsec    0.000
vs      0.000
am      0.000
gear    0.000
carb   -0.000
dtype: float64

In [6]:
from sklearn.metrics import mean_squared_error

mean_squared_error(y, estimator.predict(X))

6.431499377712996

In [7]:
import os
import pickle

os.makedirs("../submission", exist_ok=True)

with open("../submission/estimator.pkl", "wb") as file:
    pickle.dump(estimator, file)